# Section 6 · AgentCore Gateway — Turn an Existing REST API into Agent Tools (Groq edition)

> **Groq edition.** Same lab as `gateway_openapi_target.ipynb`; only the agent's model runs on Groq. AgentCore Gateway, AgentCore Identity, S3 and IAM are AWS services, so you still need AWS credentials — but no Bedrock model access. Set `GROQ_API_KEY` in your environment or a `.env` file.

Most companies already have REST APIs. **AgentCore Gateway** reads an API's OpenAPI spec and exposes every operation as an **MCP tool** — no wrapper code. Any MCP-capable agent can then discover and call those tools.

```
Strands agent ──MCP──► AgentCore Gateway ──HTTPS──► REST API (Open-Meteo weather)
                          │
                          └─ reads the OpenAPI spec from S3 and creates the tools
```

In this notebook you will:
1. Upload an OpenAPI spec to S3
2. Store the API's credential in AgentCore Identity
3. Create an IAM role for the gateway
4. Create the gateway (MCP protocol)
5. Register the REST API as a gateway target from the S3 spec
6. Connect a Strands agent to the gateway and use the tools
7. Clean up

**Prerequisites:** AWS credentials with permissions for `bedrock-agentcore:*` (gateway and identity), `iam:CreateRole/PutRolePolicy/PassRole/DeleteRole*`, `s3:*` on the demo bucket, and `secretsmanager` (AgentCore Identity stores the key there).

In [ ]:
%pip install -q 'strands-agents[openai]>=1.60.0' python-dotenv 'mcp>=1.23.0,<2.0.0' boto3

In [ ]:
import json
import os
import time
import uuid

import boto3

from dotenv import load_dotenv

load_dotenv()  # reads GROQ_API_KEY from a .env file, if present

MODEL_ID = os.getenv("GROQ_MODEL_ID", "openai/gpt-oss-120b")
REGION = os.getenv("AWS_REGION", os.getenv("AWS_DEFAULT_REGION", "us-east-1"))

session = boto3.Session(region_name=REGION)
account_id = session.client("sts").get_caller_identity()["Account"]
s3 = session.client("s3")
iam = session.client("iam")
control = session.client("bedrock-agentcore-control")

suffix = uuid.uuid4().hex[:6]
GATEWAY_NAME = f"weather-gateway-{suffix}"
TARGET_NAME = "WeatherAPI"
BUCKET = f"gateway-specs-{account_id}-{REGION}"
SPEC_KEY = "weather_openapi.json"
ROLE_NAME = f"AgentCoreGatewayDemoRole-{suffix}"
API_KEY_PROVIDER = f"weather-api-key-{suffix}"

print(f"Account {account_id} | Region {REGION} | Gateway {GATEWAY_NAME}")

## Step 1: Upload the OpenAPI spec to S3

`weather_openapi.json` describes one operation of the free Open-Meteo API: `getForecast(latitude, longitude, ...)`.

In [ ]:
try:
    if REGION == "us-east-1":
        s3.create_bucket(Bucket=BUCKET)
    else:
        s3.create_bucket(Bucket=BUCKET, CreateBucketConfiguration={"LocationConstraint": REGION})
except s3.exceptions.BucketAlreadyOwnedByYou:
    pass

s3.upload_file("weather_openapi.json", BUCKET, SPEC_KEY)
SPEC_URI = f"s3://{BUCKET}/{SPEC_KEY}"
print("Spec uploaded:", SPEC_URI)

## Step 2: Store the API credential in AgentCore Identity

OpenAPI targets need an outbound credential — the gateway attaches it to every call it makes to your API. Real APIs use a real key (or OAuth). Open-Meteo is free and ignores most extra headers, so a placeholder key works here — sent as `X-Demo-Key`, because Open-Meteo redirects any request that carries an `X-Api-Key` header to its paid API, and the gateway can't follow that redirect. The key is stored in AgentCore Identity (backed by Secrets Manager) — never in your code or the spec.

In [ ]:
provider = control.create_api_key_credential_provider(
    name=API_KEY_PROVIDER,
    apiKey="demo-key-open-meteo-does-not-need-one",
)
API_KEY_PROVIDER_ARN = provider["credentialProviderArn"]
print("Credential provider:", API_KEY_PROVIDER_ARN)

## Step 3: IAM role for the gateway

The gateway assumes this role to read its own configuration, fetch the API key, and read the spec.

In [ ]:
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
        "Action": "sts:AssumeRole",
        "Condition": {
            "StringEquals": {"aws:SourceAccount": account_id},
            "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:*"},
        },
    }],
}

permissions = {
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": ["bedrock-agentcore:GetGateway"],
         "Resource": f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:gateway/*"},
        {"Effect": "Allow", "Action": ["bedrock-agentcore:GetResourceApiKey", "bedrock-agentcore:GetWorkloadAccessToken"],
         "Resource": [f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:token-vault/default",
                      f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:token-vault/default/*",
                      f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:workload-identity-directory/*"]},
        {"Effect": "Allow", "Action": ["secretsmanager:GetSecretValue"],
         "Resource": f"arn:aws:secretsmanager:{REGION}:{account_id}:secret:bedrock-agentcore-identity!*"},
        {"Effect": "Allow", "Action": ["s3:GetObject"], "Resource": f"arn:aws:s3:::{BUCKET}/*"},
    ],
}

role_arn = iam.create_role(RoleName=ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_policy))["Role"]["Arn"]
iam.put_role_policy(RoleName=ROLE_NAME, PolicyName="GatewayPermissions", PolicyDocument=json.dumps(permissions))
print("Role:", role_arn)
time.sleep(10)  # let IAM propagate

## Step 4: Create the gateway

The gateway speaks **MCP** to agents. Who may call it is set by `authorizerType`:

| `authorizerType` | Callers authenticate with | Use for |
|---|---|---|
| `CUSTOM_JWT` | OAuth tokens (Cognito, Okta, Entra ID, Auth0) | user-facing apps |
| `AWS_IAM` | SigV4-signed requests with IAM credentials | service-to-service inside AWS |
| `NONE` | nothing — anyone with the URL | short-lived demos only |

The production pattern looks like this:

```python
gateway = control.create_gateway(
    name="enterprise-tools",
    roleArn=role_arn,
    protocolType="MCP",
    authorizerType="AWS_IAM",
)
```

To keep the client code simple in class, we create the gateway with `NONE` and **delete it at the end of the notebook**.

In [ ]:
def wait_for(get_fn, ready=("READY",), failed=("FAILED", "CREATE_UNSUCCESSFUL", "UPDATE_UNSUCCESSFUL"), label=""):
    while True:
        status = get_fn()["status"]
        print(f"  {label} status: {status}")
        if status in ready:
            return
        if status in failed:
            raise RuntimeError(f"{label} failed: {get_fn()}")
        time.sleep(5)


gateway = control.create_gateway(
    name=GATEWAY_NAME,
    roleArn=role_arn,
    protocolType="MCP",
    authorizerType="NONE",
    description="Demo gateway that exposes a REST API as MCP tools",
)
GATEWAY_ID = gateway["gatewayId"]
wait_for(lambda: control.get_gateway(gatewayIdentifier=GATEWAY_ID), label="Gateway")

GATEWAY_URL = control.get_gateway(gatewayIdentifier=GATEWAY_ID)["gatewayUrl"]
print("Gateway URL:", GATEWAY_URL)

## Step 5: Register the REST API as a target

Point the gateway at the spec in S3. It reads the spec and creates one tool per operation, named `<TargetName>___<operationId>`.

In [ ]:
target = control.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name=TARGET_NAME,
    description="Open-Meteo weather forecast API",
    targetConfiguration={
        "mcp": {
            "openApiSchema": {
                "s3": {"uri": SPEC_URI}
            }
        }
    },
    credentialProviderConfigurations=[{
        "credentialProviderType": "API_KEY",
        "credentialProvider": {
            "apiKeyCredentialProvider": {
                "providerArn": API_KEY_PROVIDER_ARN,
                "credentialParameterName": "X-Demo-Key",  # Open-Meteo redirects any request with X-Api-Key
                "credentialLocation": "HEADER",
            }
        },
    }],
)
TARGET_ID = target["targetId"]
wait_for(lambda: control.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID), label="Target")

## Step 6: Connect a Strands agent to the gateway

The gateway is just an MCP server over streamable HTTP. The agent discovers the tools and decides when to use them.

In [ ]:
from mcp.client.streamable_http import streamablehttp_client
from strands import Agent
from strands.models.openai import OpenAIModel
from strands.tools.mcp import MCPClient

model = OpenAIModel(
    client_args={
        "api_key": os.environ["GROQ_API_KEY"],
        "base_url": os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),  # or a local OpenAI-compatible server
        "max_retries": 5,  # retry dropped connections (the client default is 2)
    },
    model_id=MODEL_ID,
    # Groq reasoning models stream their thinking by default; hide it so only the answer prints.
    params={"extra_body": {"include_reasoning": False}},
)

gateway_client = MCPClient(lambda: streamablehttp_client(GATEWAY_URL))

with gateway_client:
    tools = gateway_client.list_tools_sync()
    print("Tools exposed by the gateway:", [t.tool_name for t in tools])

    agent = Agent(
        model=model,
        tools=tools,
        system_prompt="You are a travel assistant. Use the weather tools for any weather question.",
    )
    agent("I'm visiting Bangalore and Mumbai this week. What's the forecast for the next 3 days in each city?")

## Step 7: Clean up

Deletes the target, gateway, credential provider, IAM role and S3 bucket.

In [ ]:
control.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
while control.list_gateway_targets(gatewayIdentifier=GATEWAY_ID).get("items"):
    time.sleep(3)
control.delete_gateway(gatewayIdentifier=GATEWAY_ID)
print("Deleted gateway", GATEWAY_ID)

control.delete_api_key_credential_provider(name=API_KEY_PROVIDER)
print("Deleted credential provider", API_KEY_PROVIDER)

iam.delete_role_policy(RoleName=ROLE_NAME, PolicyName="GatewayPermissions")
iam.delete_role(RoleName=ROLE_NAME)
print("Deleted role", ROLE_NAME)

s3.delete_object(Bucket=BUCKET, Key=SPEC_KEY)
s3.delete_bucket(Bucket=BUCKET)
print("Deleted bucket", BUCKET)

## What you just did

You turned an existing REST API into agent tools **without writing any tool code**: the gateway read an OpenAPI spec, created MCP tools, injected the API credential on every call, and served the tools to a Strands agent.

The same gateway can also expose Lambda functions, Smithy models and other MCP servers as tools — and, in Section 7, enforce Cedar policies on every tool call.